In [125]:
import pandas as pd
import numpy as np
import datetime
import string
from dateutil.relativedelta import relativedelta
from dl_client import DatalakeClient

client = DatalakeClient()

In [126]:
def convert_visitcode_to_int(value: string) -> int or string:
    """
    Convert the visitcode column to integer.
    """

    # If the value is 'sc' or 'f', return the value
    if value == 'sc' or value == 'f':
        return value
    
    # If the value is 'bl', return 0
    if value == 'bl':
        return 0

    # If the value is 'm', return the month
    if value[0] == 'm':
        return int(value.split('m')[1])
    
    print('Ops, qualche caso non è stato considerato')
    return -1

In [127]:
def drop_if_all_none(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """
    Remove rows from dataframe where all values in the specified columns are None/NaN.
    """
    # Verify that all specified columns exist in the dataframe
    for col in columns:
        if col not in df.columns:
            raise ValueError(f"Column '{col}' not found in dataframe")
    
    # Create a mask for rows where all specified columns are NaN
    mask = df[columns].isna().all(axis=1)
    
    # Return dataframe with those rows dropped
    return df[~mask].copy()

In [128]:
def handle_null_viscode2(df: pd.DataFrame, columns_to_check: list, date_column: str) -> pd.DataFrame:
    """
    Process rows where VISCODE2 is null:
    - If all values in specified columns are null, remove the row
    - Otherwise, calculate the correct month based on the previous visit date of the patient
    """
    # Create a copy of the dataframe to avoid modifying the original
    result_df = df.copy()
    
    # Verify that all required columns exist in the dataframe
    required_cols = columns_to_check + [date_column, 'VISCODE2', 'PTID']
    for col in required_cols:
        if col not in result_df.columns:
            raise ValueError(f"Column '{col}' not found in dataframe")
    
    # Convert date column to datetime if it isn't already
    result_df[date_column] = pd.to_datetime(result_df[date_column])
    
    # Get rows with null VISCODE2
    null_viscode_mask = result_df['VISCODE2'].isna()
    rows_to_process = result_df[null_viscode_mask].copy()
    
    # Create a mask for rows to drop (all values in columns_to_check are null)
    drop_mask = rows_to_process[columns_to_check].isna().all(axis=1)
    
    # Create a list to collect rows to keep with updated VISCODE2
    rows_to_update = []
    
    # Process rows where not all columns are null
    for idx, row in rows_to_process[~drop_mask].iterrows():
        patient_id = row['PTID']
        visit_date = row[date_column]
        
        # Get all visits for this patient, sorted by date
        patient_visits = result_df[result_df['PTID'] == patient_id].sort_values(by=date_column)
        
        # Find the previous visit (if any)
        prev_visits = patient_visits[patient_visits[date_column] < visit_date]
        
        if len(prev_visits) > 0:
            # Get the most recent previous visit
            prev_visit = prev_visits.iloc[-1]
            
            if pd.notna(prev_visit['VISCODE2']):
                # Calculate date difference in months
                date_diff = (visit_date - prev_visit[date_column]).days / 30.44  # Average days per month
                
                if prev_visit['VISCODE2'] == 'bl' or prev_visit['VISCODE2'] == 'sc' or prev_visit['VISCODE2'] == 'f': 
                    # If previous visit was baseline, calculate months since baseline
                    new_viscode = f"m{int(round(date_diff))}"
                else:
                    # If previous visit had mXX format, add the months
                    prev_month = int(prev_visit['VISCODE2'].replace('m', ''))
                    new_viscode = f"m{int(round(prev_month + date_diff))}"
                
                # Update the row's VISCODE2
                result_df.loc[idx, 'VISCODE2'] = new_viscode
            else:
                # If previous visit also had null VISCODE2, we can't reliably calculate
                # Keep the row but leave VISCODE2 as null
                pass
        else:
            # No previous visits, might be baseline
            result_df.loc[idx, 'VISCODE2'] = 'bl'
    
    # Filter out rows with all null values in specified columns and null VISCODE2
    rows_to_drop = rows_to_process[drop_mask].index
    result_df = result_df.drop(rows_to_drop)
    
    return result_df

In [129]:
def replace_unknown_values(df: pd.DataFrame) -> pd.DataFrame:
    """
    Replace 'Unknown', 'unknown', and '-4' values with NaN across all columns in a dataframe.
    """
    # Create a copy of the dataframe to avoid modifying the original
    result_df = df.copy()
    
    # Dictionary of values to replace with NaN
    replace_dict = {
        'Unknown': np.nan,
        'unknown': np.nan,
        '-4': np.nan
    }
    
    # Replace values across the entire dataframe
    result_df = result_df.replace(replace_dict)
    
    # Handle numeric columns where -4 might be stored as an integer
    for col in result_df.select_dtypes(include=['number']).columns:
        result_df[col] = result_df[col].replace(-4, np.nan)
    
    return result_df

# Età

In [156]:
def add_calculated_age(exam_date, birth_date=None, birth_year=None, age_bl=None, bl_date=None, visit_code=None):
    exam_date = pd.to_datetime(exam_date, errors='coerce').date()
    
    if pd.notna(birth_date):
        birth_date = pd.to_datetime(birth_date, errors='coerce').date()
        diff = relativedelta(exam_date, birth_date)
        age = round(diff.years + (diff.months/12),1)
        return age
    
    if pd.notna(birth_year):
        birth_year = pd.to_datetime(birth_year, errors='coerce').date()
        age = float(exam_date.year - birth_year)
        return age
    
    if pd.notna(age_bl) and pd.notna(bl_date):
        age_bl = float(age_bl)
        bl_date = pd.to_datetime(bl_date, errors='coerce').date()
        diff = relativedelta(exam_date, bl_date)
        age = round(age_bl + (diff.years + (diff.months/12)),1)
        return age
    
    if pd.notna(age_bl) and pd.notna(visit_code):
        age_bl = float(age_bl)
        age = age_bl + visit_code/12
        return age

    else:
        age = None
        return age

In [132]:
def age_when_missing_bl_date(df, ID, ID_col, AGE_col, visit_date_col, visit_code_col):
    index_bl = df[(df[ID_col] == ID) & (df[visit_code_col] == 0)].index[0]
    age_bl = df[AGE_col][index_bl]
    if pd.notna(age_bl):
        bl_date = df[visit_date_col][index_bl]
        if pd.notna(bl_date):
            for index in df[df[AGE_col].isna()].index:
                exam_date=df[visit_date_col][index] 
                df[AGE_col][index] = add_calculated_age(exam_date=exam_date, age_bl=age_bl, bl_date=bl_date)
        else:
            for index in df[df[AGE_col].isna()].index:
                exam_date=df[visit_date_col][index] 
                vist_code = df[visit_code_col][index]
                df[AGE_col][index] = add_calculated_age(exam_date=exam_date, age_bl=age_bl, visit_code=vist_code)
    else:
        return df       #non ci sono modifiche età incalcolabile

    return df

# GENDER
Binarizzazione del genere dei pazienti

In [133]:
def binarization_gender(df, col_name):
    # female : 0
    # male : 1

    if df[col_name].map(type).value_counts().idxmax() == str:
        df[col_name] = df[col_name].str.strip().str.lower()  # pulizia
        df[col_name] = df[col_name].map({'female': 0, 'male': 1}).astype('Int64')
        return df

    if df[col_name].map(type).value_counts().idxmax() == int or df[col_name].map(type).value_counts().idxmax() == float:
        df[col_name] = df[col_name].apply(lambda x: 1 if x in [1, 1.0] else (0 if x in [2, 2.0] else np.nan)).astype('Int64')
        return df

# MARRY
Categorizzazione dello stato maritale

In [134]:
def categorize_marry(df, col_name):
    # Married = 1, Divorced = 2, Widowed = 3, Never married = 0, Unknown/ nan = nan

    if df[col_name].map(type).value_counts().idxmax() == str:
        df[col_name] = df[col_name].str.strip().str.lower()  # pulizia
        df[col_name] = df[col_name].map({'married' : 1, 'divorced' : 2, 'widowed' : 3, 'never married' : 0}).astype('Int64')
        return df

    if df[col_name].map(type).value_counts().idxmax() == int or df[col_name].map(type).value_counts().idxmax() == float:
        mapping = {1: 1, 1.0: 1, 2: 2, 2.0: 2, 3: 3, 3.0: 3, 4: 4, 4.0: 4, 5: 0, 5.0: 0}
        df[col_name] = df[col_name].map(mapping).astype('Int64')
        return df

# EDUCATION
Categorizzazione dgli anni di educazione

In [135]:
def categorize_education(df, col_name):
    # Married = 1, Divorced = 2, Widowed = 3, Never married = 0, Unknown/ nan = nan
    df[col_name] = df[col_name].apply(lambda x: int(x) if pd.notna(x) and x >= 0 else np.nan)
    return df

# ETHINCITY
categorizzazione etinia

In [136]:
def categorize_ethnicity(df, col_name):
    # Not Hisp/Latino = 0 , Hisp/Latino = 1, Unknown Unknown/ nan = nan

    if df[col_name].map(type).value_counts().idxmax() == str:
        df[col_name] = df[col_name].str.strip().str.lower()  # pulizia
        df[col_name] = df[col_name].map({'not hisp/latino' : 0 , 'hisp/latino' : 1}).astype('Int64')
        return df

    if df[col_name].map(type).value_counts().idxmax() == int or df[col_name].map(type).value_counts().idxmax() == float:
        df[col_name] = df[col_name].apply(lambda x: 1 if x in [1, 1.0] else (0 if x in [2, 2.0] else np.nan)).astype('Int64')
        return df

# RACE
categorizzazione razza

In [137]:
def categorize_race(df, col_name):
    # 1 = American Indian or Alaskan Native, 2 = Asian, 3 = Native Hawaiian or Other Pacific Islander, 4 = Black or African American, 5 = White, 6=More than one race, nan = Unknown

    if df[col_name].map(type).value_counts().idxmax() == str:
        df[col_name] = df[col_name].str.strip()  # pulizia
        df[col_name] = df[col_name].map({'White': 5, 'More than one': 0, 'Black': 4, 'Asian': 2, 'Am Indian/Alaskan': 1, 'Hawaiian/Other PI': 3, '5': 5, '6': 0, '4': 4, '2': 2, '1': 1, '3': 3}).astype('Int64')
        return df

    if df[col_name].map(type).value_counts().idxmax() == int or df[col_name].map(type).value_counts().idxmax() == float:
        mapping = {1: 1, 1.0: 1, 2: 2, 2.0: 2, 3: 3, 3.0: 3, 4: 4, 4.0: 4, 5: 5, 5.0: 5, 6: 0, 6.0: 0}
        df[col_name] = df[col_name].map(mapping).astype('Int64')
        return df

# DIAGNOSIS
categorizzazione della diagnosi (NC; MCI; AD/Dementia)

In [138]:
def categorize_diagnosis(df, col_name):
    # Cognitive Normal  = 0, MCI = 1, Dementia = 2

    if df[col_name].map(type).value_counts().idxmax() == str:
        df[col_name] = df[col_name].str.strip()  # pulizia
        df[col_name] = df[col_name].map({'CN' : 0 , 'MCI' : 1, 'Dementia': 3}).astype('Int64')
        return df

    if df[col_name].map(type).value_counts().idxmax() == int or df[col_name].map(type).value_counts().idxmax() == float:
        mapping = {1: 0, 1.0: 0, 2: 1, 2.0: 1, 3: 2, 3.0: 2}
        df[col_name] = df[col_name].map(mapping).astype('Int64')
        return df

# Applicare le funzioni alle popolazioni

## ADNI

In [ ]:
df = client.download_file(
    object_name="raw/ADNIMERGE_06Jun2025.csv",
)
#display(df)
df_new = df.copy(deep=True)


> **ETA** Ha l'età alla baseline e la data alla baseline. Quindi ho tutte le informazioni di cui ho bisogno sulla stessa riga

In [ ]:
df_new.rename(columns={'AGE': 'AGE_bl'}, inplace=True)
df_new['VISCODE'] = df_new['VISCODE'].apply(lambda x: convert_visitcode_to_int(x))
df_new['AGE'] = df_new.apply(lambda row: add_calculated_age(exam_date=row['EXAMDATE'], age_bl=row['AGE_bl'], bl_date=row['EXAMDATE_bl']), axis=1)


In [ ]:

for ID in df_new['RID'][df_new[df_new['AGE'].isna()].index].unique():
    age_when_missing_bl_date(df_new, ID, ID_col='RID', AGE_col='AGE', visit_date_col='EXAMDATE', visit_code_col='VISCODE')

In [ ]:
df_new[df_new['AGE'].isna()]['RID'].unique()

In [ ]:
df_new[df_new['RID']==6832]

> **GENERE** in originale 'male' e 'fmale' ==>> trasformato in female = 0 e male = 1

In [ ]:
df_new = binarization_gender(df_new, col_name='PTGENDER')

> **MARRY** in origine le categorie sono stringhe: Married, Divorced, Widowed, Never married, Unknown, nan. La conversione sarà la seguente ==>> Married = 1, Divorced = 2, Widowed = 3, Never married = 0, Unknown/ nan = nan

In [ ]:
df_new = categorize_marry(df_new, col_name='PTMARRY')

> **EDUCATION** trasformare in intero gli anni di studio (possibile da usare come categorizzazione) e rendo nan qualsiasi valore nullo o negativo

In [ ]:
df_new = categorize_education(df_new, col_name='PTEDUCAT')

> **ETHINCITY** Valori originali  Hispanic or Latino,  Not Hispanic or Latino ==> categorizzazione 0 = Not Lat/Hisp e 1 = Hips/Lat

In [ ]:
df_new = categorize_ethnicity(df_new, col_name='PTETHCAT')

> **RACE** valori originali White, More than one, Black, Asian, American Indian or Alaskan Native, Hawaiian/Other PI, Unknown\
==> ri categorizzazione 1 = American Indian or Alaskan Native, 2 = Asian, 3 = Native Hawaiian or Other Pacific Islander, 4 = Black or African American, 5 = White, 6=More than one race, nan = Unknown

In [ ]:
df_new = categorize_race(df_new, col_name='PTRACCAT')

> **DIAGNOSIS** valori originali: NC, MCI, AD ==>> categorizzati NC = 0, MCI = 1 , AD = 2

In [ ]:
df_new = categorize_diagnosis(df_new, col_name='DX')

## PTDEMOG

In [163]:
df = client.download_file(
    object_name="raw/PTDEMOG_06Jun2025.csv",
)
display(df)
df_new = df.copy(deep=True)

,PHASE,PTID,RID,VISCODE,VISCODE2,VISDATE,PTSOURCE,PTGENDER,PTDOB,PTDOBYY,...,PTBIRPR,PTBIRGR,ID,SITEID,USERDATE,USERDATE2,DD_CRF_VERSION_LABEL,LANGUAGE_CODE,HAS_QC_ERROR,update_stamp
0,ADNI1,011_S_0002,2,sc,sc,2005-08-17,1.0,1.0,04/1931,1931.0,...,NaN,NaN,18,107,2005-08-17,NaN,NaN,NaN,NaN,2005-08-17 00:00:00.0
1,ADNI1,022_S_0001,1,f,f,2005-08-18,1.0,2.0,12/1944,1944.0,...,NaN,NaN,20,10,2005-08-18,NaN,NaN,NaN,NaN,2005-08-18 00:00:00.0
2,ADNI1,011_S_0003,3,sc,sc,2005-08-18,1.0,1.0,05/1924,1924.0,...,NaN,NaN,22,107,2005-08-18,NaN,NaN,NaN,NaN,2005-08-18 00:00:00.0
3,ADNI1,022_S_0004,4,sc,sc,2005-08-18,1.0,1.0,01/1938,1938.0,...,NaN,NaN,24,10,2005-08-18,NaN,NaN,NaN,NaN,2005-08-18 00:00:00.0
4,ADNI1,011_S_0005,5,sc,sc,2005-08-23,1.0,1.0,12/1931,1931.0,...,NaN,NaN,26,107,2005-08-23,NaN,NaN,NaN,NaN,2005-08-23 00:00:00.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6042,ADNI4,082_S_10809,10809,4_sc,sc,2025-06-03,1.0,2.0,08/1945,1945.0,...,1.0,0.0,96846,82,2025-06-04,2025-06-04,v1,e,0.0,2025-06-05 01:44:21.0
6043,ADNI4,002_S_10814,10814,4_sc,sc,2025-06-02,1.0,1.0,03/1948,1948.0,...,1.0,0.0,96992,2,2025-06-04,2025-06-04,v1,e,1.0,2025-06-05 01:44:21.0
6044,ADNI4,052_S_6412,6412,4_m12,m78,2025-05-26,1.0,1.0,07/1951,1951.0,...,1.0,0.0,97024,52,2025-06-04,2025-06-04,v1,e,0.0,2025-06-05 01:44:21.0
6045,ADNI4,168_S_6321,6321,4_init,m72,2024-05-20,1.0,1.0,07/1945,1945.0,...,1.0,1.0,97060,389,2025-06-04,2025-06-04,v1,e,0.0,2025-06-05 01:44:21.0


> **ETA**Ha sia un parametro con l'anno di nascita che uno con mese e anno di nascita, per 187 pazienti mancano entrambi. In quel caso bisogna capire se ci sono altri metodi.

In [164]:
no_unknow_df = replace_unknown_values(df_new)


In [165]:
columns_must_be_verified = ['PTGENDER', 'PTDOB', 'PTEDUCAT', 'PTETHCAT', 'PTRACCAT', 'PTADDX']

In [166]:
no_none_df = handle_null_viscode2(no_unknow_df, columns_must_be_verified, 'VISDATE')
no_none_df = drop_if_all_none(no_none_df, columns_must_be_verified)

In [167]:
no_none_df['VISCODE2'] = no_none_df['VISCODE2'].apply(lambda x: convert_visitcode_to_int(x))

df_prova = no_none_df[:10].copy(deep=True)
df_prova['AGE'] = df_prova.apply(lambda row: add_calculated_age(exam_date=row['VISDATE'],birth_date=row['PTDOBYY'], birth_year=row['PTDOB']), axis=1)


In [ ]:

for ID in no_none_df['RID'][no_none_df[no_none_df['AGE'].isna()].index].unique():
    age_when_missing_bl_date(no_none_df, ID, ID_col='RID', AGE_col='AGE', visit_date_col='VISDATE', visit_code_col='VISCODE2')  ## probaile etichette da cambiare per quando saranno tutte uguali

In [ ]:
no_none_df['VISCODE2'] = no_none_df['VISCODE2'].apply(lambda x: convert_visitcode_to_int(x))

df_prova = no_none_df[:10]
df_prova['AGE'] = df_prova.apply(lambda row: add_calculated_age(exam_date=row['VISDATE'],birth_date=row['PTDOBYY'], birth_year=row['PTDOB']), axis=1)

for ID in no_none_df['RID'][no_none_df[no_none_df['AGE'].isna()].index].unique():
    age_when_missing_bl_date(no_none_df, ID, ID_col='RID', AGE_col='AGE', visit_date_col='VISDATE', visit_code_col='VISCODE2')  ## probaile etichette da cambiare per quando saranno tutte uguali

TypeError: 'int' object is not subscriptable

In [155]:
birth_date = '04/1998'
birth_date = pd.to_datetime(birth_date, errors='coerce').date()
print(birth_date, type(birth_date))

1998-04-01 <class 'datetime.date'>


In [151]:
df_prova

,PHASE,PTID,RID,VISCODE,VISCODE2,VISDATE,PTSOURCE,PTGENDER,PTDOB,PTDOBYY,...,PTBIRPR,PTBIRGR,ID,SITEID,USERDATE,USERDATE2,DD_CRF_VERSION_LABEL,LANGUAGE_CODE,HAS_QC_ERROR,update_stamp
0,ADNI1,011_S_0002,2,sc,sc,2005-08-17,1.0,1.0,04/1931,1931.0,...,NaN,NaN,18,107,2005-08-17,NaN,NaN,NaN,NaN,2005-08-17 00:00:00.0
1,ADNI1,022_S_0001,1,f,f,2005-08-18,1.0,2.0,12/1944,1944.0,...,NaN,NaN,20,10,2005-08-18,NaN,NaN,NaN,NaN,2005-08-18 00:00:00.0
2,ADNI1,011_S_0003,3,sc,sc,2005-08-18,1.0,1.0,05/1924,1924.0,...,NaN,NaN,22,107,2005-08-18,NaN,NaN,NaN,NaN,2005-08-18 00:00:00.0
3,ADNI1,022_S_0004,4,sc,sc,2005-08-18,1.0,1.0,01/1938,1938.0,...,NaN,NaN,24,10,2005-08-18,NaN,NaN,NaN,NaN,2005-08-18 00:00:00.0
4,ADNI1,011_S_0005,5,sc,sc,2005-08-23,1.0,1.0,12/1931,1931.0,...,NaN,NaN,26,107,2005-08-23,NaN,NaN,NaN,NaN,2005-08-23 00:00:00.0
5,ADNI1,022_S_0007,7,sc,sc,2005-08-25,1.0,1.0,04/1930,1930.0,...,NaN,NaN,28,10,2005-08-25,NaN,NaN,NaN,NaN,2005-08-25 00:00:00.0
6,ADNI1,022_S_0009,9,f,f,2005-08-26,1.0,2.0,01/1924,1924.0,...,NaN,NaN,30,10,2005-08-26,NaN,NaN,NaN,NaN,2005-08-26 00:00:00.0
7,ADNI1,011_S_0008,8,sc,sc,2005-08-29,1.0,2.0,03/1921,1921.0,...,NaN,NaN,32,107,2005-08-30,NaN,NaN,NaN,NaN,2005-08-30 00:00:00.0
8,ADNI1,011_S_0011,11,f,f,2005-09-07,1.0,2.0,08/1918,1918.0,...,NaN,NaN,34,107,2005-09-07,NaN,NaN,NaN,NaN,2005-09-07 00:00:00.0
9,ADNI1,022_S_0013,13,f,f,2005-09-09,1.0,2.0,09/1947,1947.0,...,NaN,NaN,36,10,2005-09-09,NaN,NaN,NaN,NaN,2005-09-09 00:00:00.0


> **GENERE** in originale 'male' = 1.0 e 'fmale' = 2.0, valori nulli = -4 or nan ==>> trasformato in female = 0 e male = 1

In [ ]:
df_new = binarization_gender(df_new, col_name='PTGENDER')

> **MARRY** in origine le categorie sono float: 1.0, 2.0, 3.0, 4.0, 5.0, -4.0, nan, 6.0 (probabilmente un typo). La conversione sarà la seguente ==>> Married = 1, Divorced = 2, Widowed = 3, Never married = 0, Unknown/ nan / others = nan.

In [ ]:
df_new = categorize_marry(df_new, col_name='PTMARRY')

> **EDUCATION** trasformare in intero gli anni di studio (possibile da usare come categorizzazione) e rendo nan qualsiasi valore nullo o negativo

In [ ]:
df_new = categorize_education(df_new, col_name='PTEDUCAT')

> **ETHNICITY** valori originali 1 = Hispanic or Latino,  2 = Not Hispanic or Latino ==> categorizzazione 0 = Not Lat/Hisp e 1 = Hips/Lat

In [ ]:
df_new = categorize_ethnicity(df_new, col_name='PTETHCAT')

> **RACE** valori originali 1 = American Indian or Alaskan Native, 2 = Asian, 3 = Native Hawaiian or Other Pacific Islander, 4 = Black or African American, 5 = White, 6=More than one race, 7 = Unknown\
==> ri categorizzazione 1 : 1 (American Indian or Alaskan Native), 2: 2 (Asian), 3 : 3 (Native Hawaiian or Other Pacific Islander), 4 : 4 (Black or African American), 5 : 5 (White), 0 : 6 (More than one race), 7: nan (Unknown)

In [ ]:
df_new = categorize_race(df_new, col_name='PTRACCAT')

## ADSP_PHC_BIOMARKER

In [ ]:
df = client.download_file(
    object_name="raw/ADSP_PHC_BIOMARKER_06Jun2025.csv",
)
display(df)
df_new = df.copy(deep=True)

> **GENERE** in originale  i valori sono interi 'male' = 1 e 'fmale' = 2 valori nulli = -4 or nan ==>> trasformato in female = 0 e male = 1

In [ ]:
df_new = binarization_gender(df_new, col_name='PHC_Sex')

> **EDUCATION** trasformare in intero gli anni di studio (possibile da usare come categorizzazione) e rendo nan qualsiasi valore nullo o negativo

In [ ]:
df_new = categorize_education(df_new, col_name='PHC_Education')

> **ETHINCITY** Valori originali  1=Hispanic or Latino,  2=Not Hispanic or Latino ==> categorizzazione 0 = Not Lat/Hisp e 1 = Hips/Lat

In [ ]:
df_new = categorize_ethnicity(df_new, col_name='PHC_Ethnicity')

> **RACE** valori originali 1 = American Indian or Alaskan Native, 2 = Asian, 3 = Black, 4 = Native Hawaiian or Other Pacific Islander, 5 = White, 6=More than one race, Other, or Unknown\
==> ri categorizzazione 1 : 1 (American Indian or Alaskan Native), 2: 2 (Asian), 4 : 3 (Native Hawaiian or Other Pacific Islander), 3 : 4 (Black or African American), 5 : 5 (White), 6 : nan (More than one race, Other, or Unknown)

In [ ]:
# non funziona la funzione perche black e Native Hawaian or PI sono invertite
mapping = {1: 1, 1.0: 1, 2: 2, 2.0: 2, 4: 3, 4.0: 3, 3: 4, 3.0: 4, 5: 5, 5.0: 5}
df_new['PHC_Race'] = df_new['PHC_Race'].map(mapping)

> **DIAGNOSIS** valori originali: 1 = NC, 2 = MCI, 3 = AD ==>> categorizzati NC = 0, MCI = 1 , AD = 2

In [ ]:
df_new = categorize_diagnosis(df_new, col_name='PHC_Diagnosis')

## BLCHANGE

> **DIAGNOSIS** valori originali: 1 = NC, 2 = MCI, 3 = AD ==>> categorizzati NC = 0, MCI = 1 , AD = 2

In [ ]:
df_new = categorize_diagnosis(df_new, col_name='BCPREDX')

# DXSUM

> **DIAGNOSIS** valori originali: 1 = NC, 2 = MCI, 3 = Dementia ==>> categorizzati NC = 0, MCI = 1 , AD = 2

In [ ]:
df_new = categorize_diagnosis(df_new, col_name='DIAGNOSIS')